# 07 Dataset Packaging

Create model-ready images, labels, metadata, and glacier-level train/validation/test splits.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from igcd.config import load_config
from igcd.packaging import package_dataset

config = load_config(PROJECT_ROOT / 'config' / 'config.json')
inventory = gpd.read_file(config.paths['processed'] / 'inventory' / 'glacier_inventory.geojson')
image_paths = {}
for path in sorted(config.paths['exports'].glob('*/*_sentinel.tif')):
    parts = path.stem.split('_')
    image_paths[('_'.join(parts[:2]), int(parts[2]))] = path
label_paths = {}
for path in sorted((config.paths['processed'] / 'change_masks').glob('*_change.tif')):
    glacier_id = '_'.join(path.stem.split('_')[:2])
    label_paths[glacier_id] = path

metadata = {
    'name': config.dataset_name,
    'version': config.raw['dataset']['version'],
    'years': config.years,
    'source_collections': config.raw['earth_engine'],
    'export': config.raw['export'],
}
package_dataset(
    inventory,
    image_paths,
    label_paths,
    config.paths['dataset'],
    config.raw['splits'],
    metadata,
)